# PCA — Principal Component Analysis

In [ ]:
# Day 16 — PCA (Principal Component Analysis)
# I missed this class, so these notes go from BASIC to ADVANCED
# with extra depth on every concept.

# PCA is a DIMENSIONALITY REDUCTION technique
# It compresses your data — fewer columns, but keeps most of the information

# What we'll cover today (step by step):
#   1. What is dimensionality and why reduce it?
#   2. The intuition behind PCA (plain English, no math yet)
#   3. Prerequisites: feature scaling (StandardScaler)
#   4. The math behind PCA (covariance matrix, eigenvalues, eigenvectors)
#   5. Explained variance and cumulative variance
#   6. How to choose the number of components
#   7. Scree plot (visual method)
#   8. Using sklearn's PCA (the easy way)
#   9. Full pipeline: PCA + Logistic Regression on handwritten digits

# Libraries used today:
#   numpy, pandas, matplotlib, sklearn
#   (all pre-installed if you use Anaconda)

# Why PCA matters for me as a DevOps/MLOps engineer:
# - In MLOps, datasets often have HUNDREDS of features
# - More features = slower training, harder to debug, risk of overfitting
# - PCA is a standard preprocessing step in many ML pipelines
# - Understanding PCA helps me debug ML pipeline failures in production

## What Is Dimensionality? Why Reduce It?

In [ ]:
# DIMENSIONALITY = the number of features (columns) in your dataset

# Example:
# A dataset with columns: [height, weight, age, salary, city]
# has 5 dimensions (5 features)

# Real-world analogy:
# Think of describing a person. You COULD use 100 measurements:
#   height, weight, shoe size, arm length, neck size, waist...
# But most of these are CORRELATED — if someone is tall,
# they probably have long arms too.
# Instead of 100 measurements, maybe 3-4 'summary measurements'
# capture 95% of the variation between people.
# PCA finds those summary measurements automatically.

# WHY reduce dimensions?
# Problem 1: TOO MANY FEATURES
#   - 100+ columns makes training SLOW
#   - Hard to visualise (we can only see 2D or 3D plots)
#   - More features = more compute cost = higher cloud bill!

# Problem 2: CORRELATED FEATURES
#   - height and arm_length contain overlapping information
#   - Redundant features waste compute and can confuse the model

# Problem 3: NOISE AND REDUNDANCY
#   - Some features are just noise (random variation)
#   - PCA helps separate signal from noise
#   - Removing noise can actually IMPROVE model accuracy!

# What PCA does:
# Takes your original features (say 64) and creates NEW features (say 28)
# These new features are called 'principal components'
# Each component is a LINEAR COMBINATION of the original features
# Components are ORDERED — PC1 has the most info, PC2 next, and so on

## The PCA Pipeline (Overview)

In [ ]:
# The full PCA pipeline, step by step:

# Step 1: Load the dataset
#   -> Get your data with features (X) and labels (y)

# Step 2: EDA — cleaning, encoding, and SCALING
#   -> PCA is sensitive to scale, so we MUST standardise features
#   -> StandardScaler makes each feature have mean=0, std=1

# Step 3: Compute the covariance matrix
#   -> This tells us how features vary TOGETHER

# Step 4: Compute eigenvalues and eigenvectors
#   -> Eigenvalues tell us HOW MUCH variance each component captures
#   -> Eigenvectors tell us the DIRECTION of each component

# Step 5: Compute explained variance
#   -> What percentage of total info does each component hold?

# Step 6: Compute cumulative variance
#   -> How many components do we need for 90% or 95% of the info?

# Step 7: Finalize the number of components
#   -> Pick the number where cumulative variance crosses your threshold

# Step 8: Apply PCA and run your model
#   -> Transform data to fewer dimensions
#   -> Train a model (e.g., Logistic Regression) on the compressed data

# Let's walk through each step with real code!

## Step 1: Load the Dataset

In [ ]:
# We'll use the 'digits' dataset from sklearn
# It contains 1797 images of handwritten digits (0-9)
# Each image is 8x8 pixels = 64 features (pixel values 0-16)

# This is the SAME dataset used in the class session
# It's a classic ML dataset for learning — small but real

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()

# The dataset is a dict-like object with:
#   digits.data    -> the features (1797 samples × 64 features)
#   digits.target  -> the labels (0-9, which digit it is)
#   digits.images  -> same data but reshaped as 8×8 images

X = digits.data      # features: 1797 samples, each with 64 pixel values
y = digits.target    # labels: 0, 1, 2, ..., 9

print('X shape:', X.shape)    # (1797, 64) -> 1797 images, 64 pixels each
print('y shape:', y.shape)    # (1797,)    -> one label per image
print('Labels:', np.unique(y))  # [0 1 2 3 4 5 6 7 8 9]

In [ ]:
# Let's see what the images look like
# Each row of X is 64 numbers — we reshape to 8×8 to see the image

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    image = X[i].reshape(8, 8)          # 64 values -> 8×8 grid
    ax.imshow(image, cmap='gray')       # display as grayscale image
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')                       # hide axes for cleaner look
plt.tight_layout()
plt.show()

# Each image is small (8×8) but still has 64 features
# The goal: can we reduce 64 features to fewer, and still recognise digits?

In [ ]:
# Let's look at one sample in detail

print('First image as 64 pixel values:')
print(X[0])
print()
print('This represents the digit:', y[0])
print('Total features per sample:', X.shape[1])   # 64

## Step 2: Feature Scaling with StandardScaler

In [ ]:
# WHY SCALE?
# PCA finds directions of MAXIMUM VARIANCE
# If one feature has values 0-1000 and another has values 0-1,
# PCA will think the first feature is WAY more important (just because of scale)
# That's wrong — we want PCA to look at the PATTERN, not the scale

# StandardScaler transforms each feature to:
#   mean = 0
#   standard deviation = 1
# Formula: z = (x - mean) / std
# This puts ALL features on the SAME scale

# Real-world analogy:
# Comparing salary (in thousands) vs age (in years) without scaling
# is like comparing kilometres to metres — the numbers mislead you
# StandardScaler converts everything to the same 'unit'

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)    # fit (learn mean/std) + transform (apply)

print('Before scaling — first sample, first 5 values:', X[0][:5])
print('After scaling  — first sample, first 5 values:', X_scaled[0][:5])
print()
print('Mean of each feature after scaling (should be ~0):', X_scaled.mean(axis=0)[:3])
print('Std of each feature after scaling  (should be ~1):', X_scaled.std(axis=0)[:3])

## Step 3: Covariance Matrix

In [ ]:
# WHAT IS COVARIANCE?
# Covariance measures how two features VARY TOGETHER

# If feature A goes UP when feature B goes UP    -> positive covariance
# If feature A goes UP when feature B goes DOWN   -> negative covariance
# If they move independently                      -> covariance near 0

# The COVARIANCE MATRIX is a table where:
#   row i, column j = covariance between feature i and feature j
# For 64 features -> 64×64 matrix = 4096 covariance values
# The diagonal = variance of each feature with ITSELF

# Why does PCA need this?
# PCA wants to find directions where data varies the MOST
# The covariance matrix captures ALL the variance and co-variance info

cov_mat = np.cov(X_scaled.T)   # .T because np.cov expects features as ROWS

print('Covariance matrix shape:', cov_mat.shape)   # (64, 64)
print()
print('First 5×5 of the covariance matrix:')
print(pd.DataFrame(cov_mat).iloc[:5, :5])

## Step 4: Eigenvalues and Eigenvectors

In [ ]:
# This is the CORE MATH of PCA. Don't worry — I'll explain it simply.

# EIGENVECTORS = the DIRECTIONS of the new axes (principal components)
#   Think of them as arrows pointing in the direction of maximum spread

# EIGENVALUES = HOW MUCH variance is captured along each eigenvector
#   A bigger eigenvalue = that direction captures more information

# Analogy:
# Imagine a cloud of data points shaped like a football (rugby ball)
# The LONGEST axis of the football = eigenvector 1 (most variance)
# The NEXT longest axis = eigenvector 2 (second most variance)
# PCA finds these axes automatically from the covariance matrix

eig_vals, eig_vecs = np.linalg.eig(cov_mat)

print('Number of eigenvalues:', len(eig_vals))    # 64 (one per feature)
print()
print('Top 10 eigenvalues (sorted, largest first):')
print(sorted(eig_vals, reverse=True)[:10])
print()
print('Notice: first few eigenvalues are MUCH bigger than the rest')
print('This means the first few components capture MOST of the info!')

## Step 5: Explained Variance

In [ ]:
# EXPLAINED VARIANCE = what PERCENTAGE of total info does each component capture?

# Formula:
#   explained_variance_ratio = eigenvalue_i / sum_of_all_eigenvalues × 100

# Example: if eigenvalue_1 = 12.0 and total = 64.0
# then component 1 explains 12/64 × 100 = 18.75% of the info

total = sum(eig_vals)
print('Sum of all eigenvalues (total variance):', total)
print()

# Calculate explained variance for each component (sorted largest first)
var_exp = [(val / total) * 100 for val in sorted(eig_vals, reverse=True)]

print('Explained variance (%) — first 10 components:')
for i, v in enumerate(var_exp[:10]):
    print(f'  PC{i+1}: {v:.2f}%')

# Notice: PC1 alone captures a big chunk!
# The variance drops quickly — later components add very little

## Step 6: Cumulative Variance

In [ ]:
# CUMULATIVE VARIANCE = running total of explained variance
# It answers: 'If I keep the top N components, how much info do I retain?'

# Example:
#   PC1: 12%   -> cumulative: 12%
#   PC2:  9%   -> cumulative: 21%
#   PC3:  8%   -> cumulative: 29%
#   ...and so on until it reaches 100%

cum_var_exp = np.cumsum(var_exp)    # np.cumsum = cumulative sum

print('Cumulative explained variance — first 10 components:')
for i, v in enumerate(cum_var_exp[:10]):
    print(f'  Top {i+1} PCs: {v:.2f}%')

print()
print(f'Total components needed for 90% info: {np.argmax(cum_var_exp >= 90) + 1}')
print(f'Total components needed for 95% info: {np.argmax(cum_var_exp >= 95) + 1}')

# WHY +1? Because Python indexes start at 0
# np.argmax returns the INDEX of the first True value
# Index 0 means 1 component, index 4 means 5 components
# So we add +1 to get the actual COUNT

In [ ]:
# KEY TAKEAWAY:
# Original data had 64 features
# But we can keep ~95% of the information with FAR FEWER components!
# That means most of those 64 features were correlated or noisy
# PCA found the essential patterns and compressed the data

# Common thresholds:
# 90% -> aggressive compression, faster but might lose some detail
# 95% -> good balance (most common in practice)
# 99% -> conservative, keeps almost all info but less compression

## Step 7: Scree Plot (Visual Method)

In [ ]:
# A SCREE PLOT shows explained variance for each component
# It helps you VISUALLY decide how many components to keep

# Look for the 'elbow' — where the curve flattens out
# Components after the elbow add very little new information

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Individual explained variance
ax1.bar(range(1, len(var_exp)+1), var_exp, alpha=0.7, color='steelblue')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('Scree Plot — Individual Variance')

# Plot 2: Cumulative explained variance
ax2.plot(range(1, len(cum_var_exp)+1), cum_var_exp, 'o-', color='orange')
ax2.axhline(y=95, color='red', linestyle='--', label='95% threshold')
ax2.axhline(y=90, color='green', linestyle='--', label='90% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance (%)')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()

plt.tight_layout()
plt.show()

# From the scree plot:
# Left: first few bars are tall, rest are tiny -> most info is in first few PCs
# Right: the curve rises fast then flattens -> after the 'elbow', adding more PCs doesn't help much

## Step 8: Apply PCA Using sklearn (The Easy Way)

In [ ]:
# In practice, you DON'T compute covariance/eigenvalues manually
# sklearn's PCA does ALL of that for you in 2 lines!

# Two ways to specify components:
#   PCA(n_components=28)   -> keep exactly 28 components
#   PCA(n_components=0.95) -> keep enough components for 95% variance

from sklearn.decomposition import PCA

# Method: pass a float (0.0 to 1.0) = percentage of variance to keep
pca = PCA(n_components=0.95)                # keep 95% of the information
X_pca = pca.fit_transform(X_scaled)          # fit + transform in one step

print('Original shape:', X_scaled.shape)      # (1797, 64)
print('After PCA shape:', X_pca.shape)         # (1797, ??) -> far fewer columns!
print()
print(f'Reduced from {X_scaled.shape[1]} features to {X_pca.shape[1]} components')
print(f'Compression ratio: {X_pca.shape[1]/X_scaled.shape[1]*100:.1f}% of original')
print(f'Information retained: 95%')

In [ ]:
# What fit_transform does internally:

# fit():
#   1. Computes covariance matrix from X_scaled
#   2. Computes eigenvalues and eigenvectors
#   3. Sorts them by eigenvalue (most important first)
#   4. Keeps top N eigenvectors (enough for 95% variance)

# transform():
#   5. Projects original data onto the selected eigenvectors
#   6. Result: same number of rows, fewer columns

# The beauty: sklearn does steps 1-6 automatically
# We manually did steps 1-4 earlier to UNDERSTAND what PCA is doing
# In real work, you just use PCA(0.95).fit_transform(X_scaled)

## Step 9: Full Pipeline — PCA + Logistic Regression

In [ ]:
# Now let's use the PCA-compressed data to train a real ML model
# We'll classify handwritten digits using Logistic Regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y,                  # PCA-transformed features, labels
    test_size=0.2,             # 20% for testing, 80% for training
    random_state=42            # for reproducibility
)

print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Testing set:  {X_test.shape[0]} samples, {X_test.shape[1]} features')

In [ ]:
# Train a Logistic Regression model on the PCA-reduced data

model = LogisticRegression(max_iter=1000)     # max_iter=1000 to ensure convergence
model.fit(X_train, y_train)                    # train the model

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy on test set: {accuracy * 100:.2f}%')
print()
print('This is remarkable!')
print(f'We reduced {X.shape[1]} features to {X_pca.shape[1]} features')
print(f'And still got {accuracy * 100:.2f}% accuracy')
print('PCA compressed the data while preserving the essential patterns')

## Quick Summary — The Complete PCA Flow

In [ ]:
# PCA in 30 seconds:

# 1. SCALE your data (StandardScaler) — PCA is sensitive to feature scale
# 2. COMPUTE covariance matrix — how features vary together
# 3. FIND eigenvalues/eigenvectors — directions and amounts of variance
# 4. SORT by eigenvalue — most informative directions first
# 5. PICK top N components — use cumulative variance (90% or 95% threshold)
# 6. PROJECT data onto the top N eigenvectors — fewer dimensions, same info

# In practice (sklearn shortcut):
#   from sklearn.preprocessing import StandardScaler
#   from sklearn.decomposition import PCA
#   X_scaled = StandardScaler().fit_transform(X)
#   X_pca = PCA(n_components=0.95).fit_transform(X_scaled)

# KEY CONCEPTS:
# - Variance = information in the data
# - Principal components = new axes that capture maximum variance
# - Each PC is a linear combination of original features
# - PCs are ordered: PC1 > PC2 > PC3 > ... in terms of info
# - PCs are orthogonal (perpendicular) = no correlation between them!

# WHEN TO USE PCA:
# ✓ Dataset has many features (50+)
# ✓ Features are correlated with each other
# ✓ You want faster training
# ✓ You want to visualise high-dimensional data (reduce to 2D/3D)

# WHEN NOT TO USE PCA:
# ✗ You need to interpret which original features are important
#   (PCA creates new abstract features — hard to explain to business)
# ✗ Dataset has very few features
# ✗ Features are already independent (low correlation)

In [ ]:
# Practice Questions:

# 1. Load the digits dataset again. This time use PCA with n_components=0.90
#    How many components does it keep? What accuracy do you get?
#    Compare with the 95% result — is the accuracy much worse?

# 2. Try the full pipeline WITHOUT PCA (use all 64 features directly)
#    Compare the accuracy to the PCA version
#    Which trains faster? (Hint: use %%time in Jupyter to measure)

# 3. Change PCA to keep exactly 2 components: PCA(n_components=2)
#    Plot the data as a 2D scatter plot:
#      plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=5)
#    Can you visually see the digit clusters?
#    What accuracy do you get with only 2 components?

# 4. Create a scree plot for the digits dataset (we did this above)
#    Where is the 'elbow'? Does it match the 90%/95% threshold?

# 5. Explain in your own words:
#    a) Why must we scale BEFORE PCA?
#    b) What does an eigenvalue represent?
#    c) Why does PCA order components by explained variance?

### DevOps / AI-MLOps Practice

In [ ]:
# PCA is a critical step in many ML pipelines I'll build and maintain
# Understanding it helps me debug pipeline failures and optimise costs

# Exercise 1: MLOps Pipeline Monitoring (conceptual)
# You're maintaining an ML pipeline that processes server logs:
#   - Input: 150 features per log entry (CPU, memory, disk I/O, network...)
#   - Model: anomaly detection for security alerts
#   - Problem: training takes 45 minutes, retraining happens daily
#
# Your team lead asks you to speed up training.
# Write pseudocode for adding PCA to the pipeline:
#   a) Where in the pipeline should PCA go? (before/after scaling?)
#   b) What threshold would you start with? (90%? 95%? 99%?)
#   c) If PCA reduces 150 features to 30, estimate the speedup
#   d) What metric would you check to ensure quality didn't drop?

# Exercise 2: Hands-On — Server Metrics PCA
# Create a FAKE dataset of server metrics:
#
# import numpy as np
# np.random.seed(42)
# n_servers = 500
# cpu_usage = np.random.uniform(10, 95, n_servers)
# memory_usage = cpu_usage * 0.8 + np.random.normal(0, 5, n_servers)  # correlated!
# disk_io = cpu_usage * 0.5 + np.random.normal(0, 10, n_servers)     # correlated!
# network_in = np.random.uniform(100, 5000, n_servers)               # independent
# network_out = np.random.uniform(50, 3000, n_servers)               # independent
# temperature = cpu_usage * 0.3 + np.random.normal(25, 3, n_servers) # correlated!
#
# X_server = np.column_stack([cpu_usage, memory_usage, disk_io,
#                             network_in, network_out, temperature])
#
# a) Scale the data with StandardScaler
# b) Apply PCA with 0.95 variance threshold
# c) How many components remain? (expect fewer than 6!)
# d) Create a scree plot
# e) Explain WHY PCA reduced the features
#    (hint: cpu, memory, disk_io, temperature are correlated)

# Exercise 3: Debugging a PCA Pipeline
# A junior ML engineer sends you this pipeline and says 'accuracy is terrible':
#
#   from sklearn.decomposition import PCA
#   from sklearn.linear_model import LogisticRegression
#   X_pca = PCA(n_components=0.95).fit_transform(X)   # <-- spot the bug!
#   model = LogisticRegression().fit(X_pca, y)
#
# What's missing? (Hint: one critical step before PCA)
# Fix the pipeline and explain WHY the fix matters.

# Why PCA matters for me as a DevOps/MLOps engineer:
# - Faster training = lower compute costs on Azure ML / AKS
# - Smaller models = faster inference = better latency for APIs
# - Understanding PCA helps me debug 'model degradation' issues
# - PCA is used in anomaly detection for monitoring (real DevOps use case!)